## **Engenharia de recursos e sele??o de vari?veis**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi']=100

In [ ]:
#Configura??o para poder apresentar todas as linhas
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

In [ ]:
X_dados_balanceados = pd.read_csv('X_dados_balanceados.csv')

In [ ]:
y_dados_balanceados = pd.read_csv('y.csv')

Lembrar que, uma grande quantidade de vari?veis de entrada n?o significa bom desempenho e talvez seja at? ruim para o modelo.

Visto isso, com base na an?lise explorat?ria e t?cnicas de sele??o de vari?veis, vamos selecionar algumas vari?veis especificas.

Como exemplo, talvez n?o seja necess?rio utilizar todas as colunas de profiss?o e educa??o que tornaram-se bin?rias, talvez seja interessante trabalhar apenas com aquelas que possuem mais impacto, como pode ser o caso das colunas de profiss?o e educa??o mais altas, tendo em vista que as pessoas que tem profiss?o e educa??o mais relevantes, tenham mais chances de aderir a campanha.

## **Engenharia de recursos**

J? realizamos algumas engenharia de recursos ao transformar colunas cat?goricas em colunas bin?rias, e a partir disso agora tamb?m pode-se avaliar quais s?o relevantes e quais devem ser descartadas.

Com base no que foi analisado na an?lise explorat?ria, pode-se pensar em algumas outras vari?veis que podem ser criadas atrav?s das vari?veis que tem mais impacto na ader?ncia da campanha, sendo o caso das vari?veis:

* Dura??o
* Indice de confian?a do consumidor
* Campanha anterior

In [ ]:
sucesso_aderencia_campanha = X_dados_balanceados.query('aderencia_campanha == 1')
sucesso_aderencia_campanha.head()

### **Vari?vel dura??o**

In [ ]:
sns.distplot(sucesso_aderencia_campanha['duracao'], bins=25)

Percebe-se que entre 100 a 1000 existe uma alta ader?ncia a campanha, ent?o o que irei fazer ? construir uma vari?vel com base nesse intervalo, para que esse intervalo seja entendido como um periodo de sucesso da ader?ncia a campanha.

In [ ]:
duracao_100_1000 = X_dados_balanceados.query("duracao > 100 & duracao < 1000")
duracao_100_1000.aderencia_campanha.value_counts()

Note que o valor de ader?ncia a campanha acaba sendo maior do que o da n?o ader?ncia da campanha.

In [ ]:
def sucesso_duracao(duracao):
    if (duracao > 100 and duracao < 1000):
        return 1
    else:
        return 0 

In [ ]:
X_dados_balanceados['duracao_bem_sucedida'] = X_dados_balanceados.apply(lambda row: sucesso_duracao(row['duracao']), axis='columns')

Se os dados est?o no intervalo de dura??o que foi determinado ? retornada a classe bin?ria 1 (como sendo um sucesso).

In [ ]:
#Visualizando como os dados a partir do sucesso da regra de dura??o se comparta quando ? aderida a campanha
X_dados_balanceados[(X_dados_balanceados['duracao_bem_sucedida']== 1) & (X_dados_balanceados['aderencia_campanha']==1)].head(5)

### **Vari?vel campanha anterior**

In [ ]:
X_dados_balanceados.head()

In [ ]:
X_dados_balanceados.campanha_anterior.value_counts()

In [ ]:
X_dados_balanceados.groupby(['campanha_anterior'])['aderencia_campanha'].sum().sort_values(ascending=False)

Observa-se que como j? mencionado na an?lise explorat?ria, o fato da campanha anterior n?o existir pode ter forte influ?ncia no resultado da ader?ncia da campanha, por ser a maior classe da vari?vel.

Ent?o o que irei fazer ? criar uma vari?vel em que a campanha anterior n?o tenha existido e que o periodo de dura??o da campanha seja no intervalo de 100 a 1000 determinado anteriormente.

In [ ]:
semCampanhaAnterior_interDuracao = X_dados_balanceados.query("campanha_anterior == 2 & duracao_bem_sucedida == 1")
semCampanhaAnterior_interDuracao.aderencia_campanha.value_counts()

In [ ]:
def SemCampanhaAnterior_intervalo_duracao(campanha_anterior, intervalo_duracao):
    if (campanha_anterior == 2 and intervalo_duracao == 1):
        return 1
    else:
        return 0 

In [ ]:
X_dados_balanceados['semCampanhaAnterior_interDuracao'] = X_dados_balanceados.apply(lambda row: SemCampanhaAnterior_intervalo_duracao(row['campanha_anterior'], row['duracao_bem_sucedida']), axis='columns')

In [ ]:
#Visualizando como os dados a partir dessa nova vari?vel se comporta a partir da ader?ncia da campanha
X_dados_balanceados[(X_dados_balanceados['semCampanhaAnterior_interDuracao']== 1) & (X_dados_balanceados['aderencia_campanha']==1)].head(5)

In [ ]:
X_dados_balanceados[['aderencia_campanha', 'semCampanhaAnterior_interDuracao', 'duracao_bem_sucedida']].corr()

Como pode-se observar a regra de vari?vel criada para a quando n?o existe campanha anterior no intervalo de dura??o de 100 a 1000, possivelmente n?o faz tanto sentido para a ader?ncia da campanha, visto a baixa correla??o. Por outro lado, a vari?vel de intervalo de dura??o constru?da possui uma correla??o aceit?vel com a a ader?ncia da campanha.

## **Sele??o de vari?veis**

A ideia dessa etapa ? diminuir o n?mero de vari?veis do conjunto de dados e manter somente aquelas vari?veis que s?o relevantes.

Talvez n?o seja necess?rio utilizar todas as colunas de profiss?o e educa??o que tornaram-se bin?rias, talvez seja interessante trabalhar apenas com aquelas que possuem mais impacto, como pode ser o caso das colunas de profiss?o e educa??o mais altas, tendo em vista que as pessoas que tem profiss?o e educa??o mais relevantes, tenham mais chances de aderir a campanha.

Como tamb?m foi visto atrav?s do gr?fico de correla??o, a melhor educa??o e profiss?o s?o as que acabam tendo mais correla??o com a ader?ncia da campanha, ent?o algumas vari?veis bin?rias de educa??o e profiss?oque foram criadas acaba n?o sendo t?o necess?rio.

In [ ]:
#importando os pacotes
from yellowbrick.features import FeatureImportances
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier;

Irei agora inicialmente utilizar as vari?veis de entrada e a vari?vel de sa?da que definimos anteriormente e as novas vari?veis que foram constru?das, para fazer uma sele??o dessas vari?veis e decidir quais devem permanecer, tendo em vista as que mais podem ter influ?ncia na ader?ncia da campanha.

In [ ]:
X = X_dados_balanceados.drop(["aderencia_campanha", 'taxa_juros_media', 'indice_confianca_consumidor', 'indice_precos_consumidor'],1)
y = y_dados_balanceados['aderencia_campanha']

### **Yellowbrick**

A biblioteca Yellowbrick possui o atributo FeatureImportances para classificar e plotar as import?ncias relativas das vari?veis.

In [ ]:
# Gr?fico :
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot()

#Realizando configura??o para o gr?fico:
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)

#Plotando o gr?fico:
viz = FeatureImportances(RandomForestClassifier(random_state=0), ax=ax)
viz.fit(X, y)
viz.poof();

### **Recursive feature elimination**

O Recursive Feature Elimination (RFE) ? uma t?cnica para sele??o de atributos. Basicamente ele remove recursivamente os atributos e constr?i o modelo com os atributos remanescentes. Essa t?cnica utiliza a acur?cia do modelo para identificar os atributos que mais contribuem para prever a vari?vel alvo.

O objetivo com o RFE ser? selecionar 10 vari?veis de entrada com base em todas as vari?veis presentes no conjunto de dados.

In [ ]:
#RFE com Random Forest Classifier para selecionar 10 vari?veis relevantes 
randomfc = RandomForestClassifier(random_state=0)
rfe = RFE(randomfc, n_features_to_select= 10)

In [ ]:
#Fit nos dados
rfe.fit(X, y)

Ap?s treinar O RFE com as vari?veis de entrada, pode-se verificar quais s?o as dez vari?veis mais relevantes.

In [ ]:
#Visualizando quais foram as 10 variaveis selecionadas
rfe.support_
pd.DataFrame({'coluna': X.columns,
              'bool': rfe.get_support()}).sort_values(by='bool',ascending=False).head(10).reset_index(drop=True)

**Notas**

* A vari?vel **idade** j? tinha apresentado um resultado de correla??o com a ader?ncia da campanha interessante, ent?o era esperado.


* As vari?veis **mes, dia da semana e campanha anterior** n?o apresentaram bom resultado na correla??o realizada na explora??o multivariada, por?m eu tinha mencionado que poderiam ser muito relevantes e atrav?s das duas t?cnicas aplicadas, provavelmente tem uma forte influ?ncia na ader?ncia da campanha.


* Interessante observar que para as duas t?cnicas aplicadas, a profiss?o **colarinho azul** se tornou bastante relevante para o resultado do problema que ? a ader?ncia da campanha. Mais relevante at? que a a profiss?o admin, que apresentava melhores resultados na ader?ncia da campanha. 



* Uma vari?vel que n?o foi muito analisada na explora??o dos dados e acaba tornando-se relevante ? a **dias ?ltimo contato**, o que n?o era esperado, mas faz sentido.



* As vari?veis **indice de pre?os e confian?a, taxa de juros m?dia** j? tinham sido analisadas e foram tratadas cuidadosamente, j? com a ideia de que seriam relevantes para o problema, ent?o era esperado. Como tinha mencionado inicialmente, possivelmente eram as vari?veis n?mericas mais importantes.


* A vari?vel **dura??o** foi a que mais estava claro que tinha uma forte influ?ncia no resultado positivo para o problema, e tamb?m acaba sendo uma vari?vel muito relevante na ader?ncia da campanha.


A surpresa foi que algumas vari?veis apresentaram um resultado de correla??o interessante, nas an?lises explorat?rias realizadas aparentava ter uma boa influ?ncia para o problema, por?m n?o foram entendidas como vari?veis t?o relevantes na t?cnica Yellowbrick e na t?cnica RFE n?o apareceu entre as 10 vari?veis mais relevantes, sendo elas:

 * Meio de contato
 * Quantidade de contatos totais
 
 
 Al?m disso, as vari?veis de emprestimos tamb?m acabaram ficando de lado, apesar da vari?vel emprestimo moradia n?o est? t?o ruim com a t?cnica de Yellowbrick.

## **Conclus?es**

A partir do resultado das t?cnicas de sele??o de vari?veis recursive feature elimination e Yellowbrick, j? consigo entender melhor quais vari?veis de fato s?o relevantes e definir meus dados de entrada.

Ent?o ser?o usadas como vari?veis de entrada todas as 10 vari?veis que o recursive feature elimination selecionou, tendo em vista que tamb?m aparecem nas primeiras posi??es de vari?veis relevantes da t?cnia Yellowbrick. Por?m, al?m dessas vari?veis, irei inserir algumas que com o resultado do yellowbrick e an?lise explorat?ria parecem tamb?m serem vari?veis interessantes para o problema, sendo elas:

* Ensino m?dio
* Quantidade contatos campanha
* Gradua??o completa
* Emprestimo moradia

In [ ]:
X_select = X.columns[rfe.support_]
X_select

In [ ]:
X_select = X_dados_balanceados[['idade', 'mes', 'dia_da_semana', 'duracao', 'qtd_contatos_campanha',
       'dias_ultimo_contato', 'campanha_anterior',
       'taxa_juros_media_normalizado', 'indice_precos_consumidor_normalizado',
       'indice_confianca_consumidor_normalizado']].copy()

In [ ]:
X_select[['ensino_medio', 'qtd_contatos_campanha', 'graduacao_completa', 'emprestimo_moradia']] = X_dados_balanceados[['ensino_medio', 'qtd_contatos_campanha', 'graduacao_completa', 'emprestimo_moradia']].copy()

In [ ]:
#Salvando os dados de entrada finais
X_select.to_csv("X.csv",index= False)